In [1]:
import json
from pathlib import Path

import pandas as pd

In [2]:
base_dir = Path(".").resolve()
data_dir = base_dir / "data"

In [3]:
# Credential Group
with open(data_dir / "credential_group.json") as file:
    data = json.load(file)
    
df = pd.DataFrame(data.get("clientGroupList", []))
df.rename(columns={"clientCodes": "credential_list"}, inplace=True)
df["id"] = df["id"].astype(int)

df.to_csv(data_dir / "credential_group.csv", index=False)

In [15]:
# Distribution Rules
with open(data_dir / "metadata.json") as meta_file:
    metadata = json.load(meta_file)

with open(data_dir / "distribution_rules.json") as file:
    data = json.load(file, object_hook=lambda obj: {**obj.pop("lvl", {}), **obj})
    rules = data.get("rules", [])

    for rule in rules:
        rrg = rule.get("rrg")
        if rrg is None:
            rule["rrg"] = {}

        for key in list(rule.keys()):
            if key in metadata:
                rule[metadata[key]] = rule.pop(key)

        level_mapping = metadata["level_mapping"]
        for key, value in rule.items():
            if isinstance(value, dict):
                for k in list(value.keys()):
                    if k in level_mapping:
                        value[level_mapping[k]] = value.pop(k)

df = pd.json_normalize(rules, sep="_")
df.to_csv(data_dir / "distribution_rules.csv", index=False)